# 01 - Model, observability and controllability analysis

Numerical checks for the Track A models in `estimation/models.py`. Observability is evaluated for the linearised system along a short motion, at randomly sampled robot poses and motions. These are numerical evidence at sampled poses, not a proof for all poses.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root, when run from notebooks/
import numpy as np
from estimation.models import (H_marker_pose, H_range_bearing, observability_matrix, smallest_singular_value,
                               tracking_error_model, controllability_rank)

DT = 0.1
MARKER = np.array([2.0, 1.0, 0.5])    # known marker pose in the world frame: x, y, orientation (illustrative values)
SECOND = np.array([-1.0, 3.0, 0.0])   # second marker, used only to show that extra information restores observability

In [ ]:
rng = np.random.default_rng(1)
cases = {
    "one marker, range+bearing": lambda z: H_range_bearing(z, MARKER),
    "two markers, range+bearing": lambda z: np.vstack([H_range_bearing(z, MARKER), H_range_bearing(z, SECOND)]),
    "one marker, full pose": lambda z: H_marker_pose(z, MARKER),
}
sv = {name: [] for name in cases}
for _ in range(300):
    x = np.array([rng.uniform(-3, 3), rng.uniform(-3, 3), rng.uniform(-np.pi, np.pi)])
    if min(np.linalg.norm(x[:2] - MARKER[:2]), np.linalg.norm(x[:2] - SECOND[:2])) < 0.5:
        continue
    controls = [(rng.uniform(0.05, 0.2), rng.uniform(-0.4, 0.4))] * 6
    for name, H_fn in cases.items():
        sv[name].append(smallest_singular_value(observability_matrix(x, controls, DT, H_fn)))

print("smallest singular value of the observability matrix (near 0 means unobservable)")
for name, vals in sv.items():
    vals = np.array(vals)
    print(f"{name:28s} n={len(vals)}  min {vals.min():.2e}  median {np.median(vals):.2e}")

In [ ]:
print("reference speed -> controllability rank of the tracking-error model (3 means controllable)")
for vr, wr in [(0.15, 0.0), (0.15, 0.3), (0.0, 0.3), (0.0, 0.0)]:
    A, B = tracking_error_model(vr, wr)
    print(f"v_r={vr:4.2f} w_r={wr:4.2f}  rank {controllability_rank(A, B)}")

## Reading the results

- Range and bearing to a single known marker leaves one direction unobservable at every sampled pose and motion. The direction is a rotation of the whole trajectory about the marker, which changes neither the body-frame odometry nor the range and bearing. Changing the robot heading or path does not remove it.
- Adding information that breaks this symmetry restores full rank: the marker orientation (full marker pose) or a second marker.
- The tracking-error model is controllable unless the reference speed is exactly zero in both linear and angular velocity.